# Семинар 1. Знакомство с инструментами и базовый ETL-процесс 

**Цель семинара:** Настроить рабочее окружение, освоить базовые возможности библиотек pandas и numpy, локализовать и автоматизировать загрузку многомодальных данных вашего варианта из локального контура `data/input/`, а также пошагово реализовать сквозную универсальную функцию промышленной очистки данных.

### 🔧 Настройка рабочего окружения 

Перед началом выполнения убедитесь, что ваше локальное окружение развернуто в соответствии со стандартами курса:
* Использован менеджер пакетов **UV** для управления зависимостями и версиями Python.
* В **VS Code** активны обязательные расширения `ms-python.python` и `ms-toolsai.jupyter`.
* Подключен линтер `charliermarsh.ruff` для автоматического контроля качества кода.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Настройка принудительного отображения всех колонок датафрейма 
pd.set_option('display.max_columns', None)


---

## 📥 Шаг 1. Инициализация локального контура данных 

Все исходные файлы (`profiles.csv`, `reviews.csv`, `transactions.csv`) надо положить в локальную директорию проекта `data/input/`.


In [ ]:
INPUT_DIR = os.path.join(".", "data", "input")
profiles_csv_path = os.path.abspath(os.path.join(INPUT_DIR, "profiles.csv"))

if not os.path.exists(profiles_csv_path):
    print(f"⚠️ Файл не найден по пути: {profiles_csv_path}. Для демонстрации генерируется базовый mock-датасет.")
    os.makedirs(INPUT_DIR, exist_ok=True)
    # Создание тестового контура на основе структуры полей Telco Churn 
    mock_data = pd.DataFrame({
        ' Target_ID ': ['T01', 'T02', 'T03', 'T03', 'T04'],
        'Churn': ['Yes', 'N', '1', '1', 'False'],
        'tenure': [0, 15, 40, 40, 5],
        'MonthlyCharges': ['$50.5', '  20.0', 'rub 100', 'rub 100', 'null'],
        'TotalCharges': ['?', '300', '4000.5', '4000.5', ' '],
        'gender': ['Male', 'female', 'MALE', 'MALE', 'Female'],
        'Join Date': ['2020-01-01', '2021-05-15', '2019-10-20', '2019-10-20', '2023-01-11']
    })
    mock_data.to_csv(profiles_csv_path, index=False)

print(f"Путь к CRM выгрузке успешно инициализирован: {profiles_csv_path}")


---

## 🛠 ЗАДАНИЕ 1: Импорт данных и первичный диагностический рентген 
**Задача:** Загрузите файл `profiles.csv` в переменную `df_raw`. Выведите первые 5 строк таблицы, технический паспорт (`.info()`) и базовые числовые статистики (`.describe()`).

*🤖 Застряли или не понимаете структуру? Отправьте AI-ассистенту код и соответствующий тег:*
* `#SEM1_TASK1_START` — если не знаете, с какого метода начать чтение.
* `#SEM1_TASK1_BUG` — если возникла ошибка при загрузке.
* `#SEM1_TASK1_WHY` — если вы хотите понять аномалии в типах полей.


In [ ]:
# [MASTER SOLUTION]
df_raw = pd.read_csv(profiles_csv_path)

print("--- Первые 5 строк исходного датасета ---")
print(df_raw.head())

print("\n--- Технический паспорт таблицы (info) ---")
df_raw.info()

print("\n--- Описательная статистика числовых признаков (describe) ---")
print(df_raw.describe(include='all'))


In [ ]:
# [STUDENT TEMPLATE]
# TODO: 1.1. Напишите код чтения CSV-файла с помощью библиотеки pandas 
# TODO: 1.2. Вызовите диагностические методы .head(), .info() и .describe() для анализа структуры 
raise NotImplementedError("Задание 1 не выполнено! Удалите эту строку и напишите свой код.")

df_raw = ...

print("--- Первые 5 строк исходного датасета ---")
# Ваш код для head()

print("\n--- Технический паспорт таблицы (info) ---")
# Ваш код для info()

print("\n--- Описательная статистика числовых признаков (describe) ---")
# Ваш код для describe()


---

## 🛠 ЗАДАНИЕ 2: Зачистка заголовков и удаление дубликатов 
**Задача:** Устраните технические ошибки выгрузки `ERR_DIRTY_COLUMNS` (концевые пробелы в названиях колонок, ломающие синтаксис `df.col`) и `ERR_ROW_DUPLICATE` (полные дубликаты строк из-за сбоя трансляции).

*🤖 Нужна помощь ИИ-ментора? Используйте теги:*
* `#SEM1_TASK2_START` — подсказать строковые методы `.str` для обработки индексов колонок.
* `#SEM1_TASK2_BUG` — если после удаления дубликатов размер таблицы не изменился или возникла ошибка.


In [ ]:
# [MASTER SOLUTION]
df_step2 = df_raw.copy()
df_step2.columns = df_step2.columns.str.strip().str.replace(' ', '_')
df_step2 = df_step2.drop_duplicates()

print(f"Исходный размер: {df_raw.shape} | Размер после Шага 2: {df_step2.shape}")


In [ ]:
# [STUDENT TEMPLATE]
# TODO: 2.1. Зачистите пробелы в именах колонок (.str.strip()) и замените внутренние пробелы на '_' (.str.replace()) 
# TODO: 2.2. Удалите полные дубликаты строк с помощью предназначенного метода Pandas (.drop_duplicates()) 
raise NotImplementedError("Задание 2 не выполнено! Удалите эту строку и напишите свой код.")

df_step2 = df_raw.copy()

# 1. Очистка заголовков:
df_step2.columns = ...

# 2. Удаление полных дубликатов:
df_step2 = ...

print(f"Исходный размер: {df_raw.shape} | Размер после Шага 2: {df_step2.shape}")


---

## 🛠 ЗАДАНИЕ 3: Локализация скрытых текстовых заглушек 
**Задача:** Избавьтесь от ошибок `ERR_WHITESPACE_NAN` (скрытые пробелы в ячейках) и `ERR_STRING_PLACEHOLDER` (заглушки вида `"?"`, `"null"`, `"N/A"`, `"None"`).

*🤖 Помощь AI-тьютора:*
* `#SEM1_TASK3_START` — шаблон регулярного выражения (Regex) для поиска заглушек без учета регистра.
* `#SEM1_TASK3_BUG` — если метод `.replace()` возвращает ошибку или не заменяет символы.


In [ ]:
# [MASTER SOLUTION]
df_step3 = df_step2.copy()
placeholders = [r'^\s*$', r'^\?$', r'(?i)^null$', r'(?i)^n/a$', r'(?i)^none$']

for pattern in placeholders:
    df_step3 = df_step3.replace(pattern, np.nan, regex=True)

print(f"Количество физических пропусков до очистки: {df_step2.isnull().sum().sum()}")
print(f"Количество выявленных скрытых пропусков: {df_step3.isnull().sum().sum()}")


In [ ]:
# [STUDENT TEMPLATE]
# TODO: 3.1. Изучите список регулярных выражений placeholders для поиска текстового мусора 
# TODO: 3.2. Напишите цикл, заменяющий эти заглушки на честные пустоты np.nan по всему датафрейму через .replace() с параметром regex=True 
raise NotImplementedError("Задание 3 не выполнено! Удалите эту строку и напишите свой код.")

df_step3 = df_step2.copy()

# Список регулярных выражений для поиска заглушек
placeholders = [r'^\s*$', r'^\?$', r'(?i)^null$', r'(?i)^n/a$', r'(?i)^none$']

for pattern in placeholders:
    df_step3 = ...

print(f"Количество физических пропусков до очистки: {df_step2.isnull().sum().sum()}")
print(f"Количество выявленных скрытых пропусков: {df_step3.isnull().sum().sum()}")


---

## 🛠 ЗАДАНИЕ 4: Восстановление числовых/временных типов и заполнение NaN 
**Задача:** Исправьте ошибку `ERR_NUMERIC_AS_OBJECT` и `ERR_DATE_AS_OBJECT`. Преобразуйте потенциально числовые/финансовые колонки (`TotalCharges`, `MonthlyCharges`, `Num_Feature_X`), содержащие мусорные знаки валют (`$`, `руб`) или пробелы, в тип `float`. Столбцы с датами (например, `Join_Date`) приведите к типу `datetime`. Примените бизнес-логику заполнения пустот (`ERR_NAN`): если жизненный цикл клиента `tenure == 0`, заполните пропуск в общих затратах строго `0`. Иные изолированные пропуски заполните медианой столбца.

*🤖 Помощь AI-тьютора:*
* `#SEM1_TASK4_START` — как применить `pd.to_numeric` и `pd.to_datetime` с параметром `errors='coerce'`.
* `#SEM1_TASK4_BUG` — ошибка при замене знаков валют или неверная фильтрация через `np.where`.
* `#SEM1_TASK4_WHY` — почему новые клиенты с `tenure == 0` не должны заполняться средней или медианой.


In [ ]:
# [MASTER SOLUTION]
df_step4 = df_step3.copy()

# 1. Приведение дат к datetime
datetime_cols = ['Join_Date', 'Trans_Date', 'Review_Date']
for col in datetime_cols:
    if col in df_step4.columns:
        df_step4[col] = pd.to_datetime(df_step4[col], errors='coerce')

# 2. Очистка и приведение числовых признаков
numeric_cols = ['TotalCharges', 'MonthlyCharges', 'Num_Feature_X']
for col in numeric_cols:
    if col in df_step4.columns:
        df_step4[col] = df_step4[col].astype(str).str.replace(r'[^\d.]', '', regex=True)
        df_step4[col] = pd.to_numeric(df_step4[col], errors='coerce')
        
        # Бизнес-логика: если клиент новый (tenure == 0), то TotalCharges / финансовые фичи равны 0
        if 'tenure' in df_step4.columns:
            df_step4[col] = np.where((df_step4['tenure'] == 0) & (df_step4[col].isnull()), 0.0, df_step4[col])
            
        # Заполнение остаточных пропусков медианой
        if df_step4[col].isnull().sum() > 0:
            df_step4[col] = df_step4[col].fillna(df_step4[col].median())

print("Числовые и временные типы успешно восстановлены!")


In [ ]:
# [STUDENT TEMPLATE]
# TODO: 4.1. Переведите временные столбцы (например, 'Join_Date') в формат datetime через pd.to_datetime 
# TODO: 4.2. Очистите текстовые строки числовых колонок от валютных знаков, пробелов и букв с помощью регулярного выражения r'[^\d.]' 
# TODO: 4.3. Примените метод pd.to_numeric(..., errors='coerce') для конвертации в float 
# TODO: 4.4. Реализуйте бизнес-правило через np.where: если tenure == 0, запишите 0, иначе оставьте исходное значение 
# TODO: 4.5. Остаточные NaN в числовых столбцах заполните медианой этого же столбца 
raise NotImplementedError("Задание 4 не выполнено! Удалите эту строку и напишите свой код.")

df_step4 = df_step3.copy()
numeric_cols = ['TotalCharges', 'MonthlyCharges', 'Num_Feature_X']
datetime_cols = ['Join_Date', 'Trans_Date', 'Review_Date']

# 1. Работа с датами
for col in datetime_cols:
    if col in df_step4.columns:
        df_step4[col] = ...

# 2. Работа с числами
for col in numeric_cols:
    if col in df_step4.columns:
        # Шаг 4.1: Очистка от мусорных символов и регулярное выражение
        df_step4[col] = ...
        # Шаг 4.2: Преобразование типов
        df_step4[col] = ...
        # Шаг 4.3: Логическое заполнение пропусков (tenure == 0)
        if 'tenure' in df_step4.columns:
            df_step4[col] = np.where(..., ..., df_step4[col])
        # Шаг 4.4: Заполнение остаточных NaN медианой
        df_step4[col] = ...


---

## 🛠 ЗАДАНИЕ 5: Унификация флагов и исправление регистра категорий 
**Задача:** Устраните хаос во флагах `ERR_MIXED_BOOLEAN` (смешение стилей `Yes`/`No`, `Y`/`N`, `True`/`False`, `1`/`0`). Приведите все бинарные колонки к единому типу `bool`. Дополнительно исправьте `ERR_CASE_INCONSISTENCY` (грязный регистр текстовых категорий), переведя их в единый нижний регистр (`.str.lower()`) во избежание дублирования уникальных значений при группировках.

*🤖 Помощь AI-тьютора:*
* `#SEM1_TASK5_START` — как составить списки маппинга (`true_variants` / `false_variants`) и применить метод `.isin()`.
* `#SEM1_TASK5_WHY` — почему грязный регистр категорий (например, "male" и "Male") критически ломает графики и ИИ-модели.


In [ ]:
# [MASTER SOLUTION]
df_step5 = df_step4.copy()
binary_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'Churn', 'TechSupport', 'Target_Flag', 'Exited', 'Default', 'Converted', 'Cancelled', 'Lapsed', 'Retained', 'Abandoned', 'Dropped', 'NoShow', 'Fraud', 'Stale', 'Renewed', 'Churned']
categorical_cols = ['gender', 'Gender', 'Contract', 'PaymentMethod', 'InternetService', 'Cat_Feature_Y']

true_variants = ['YES', 'Y', '1', 'TRUE', '1.0']
false_variants = ['NO', 'N', '0', 'FALSE', '0.0']

for col in binary_cols:
    if col in df_step5.columns:
        str_col = df_step5[col].astype(str).str.strip().str.upper()
        df_step5.loc[str_col.isin(true_variants), col] = True
        df_step5.loc[str_col.isin(false_variants), col] = False
        
        if df_step5[col].isnull().sum() > 0:
            df_step5[col] = df_step5[col].fillna(df_step5[col].mode()[0])
        df_step5[col] = df_step5[col].astype(bool)

for col in categorical_cols:
    if col in df_step5.columns:
        df_step5[col] = df_step5[col].astype(str).str.strip().str.lower()

print("Успешно нормализовано!")


In [ ]:
# [STUDENT TEMPLATE]
# TODO: 1. Приведите бинарные флаги из списка binary_cols к единому булевому типу True/False на основе списков true_variants/false_variants 
# TODO: 2. Заполните возможные пустоты во флагах самым частым значением (модой) 
# TODO: 3. Стандартизируйте регистр текстовых категорий из списка categorical_cols, переведя их в единый .str.lower() 
raise NotImplementedError("Задание 5 не выполнено! Удалите эту строку и напишите свой код.")

df_step5 = df_step4.copy()
binary_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'Churn', 'TechSupport', 'Target_Flag', 'Exited', 'Default', 'Converted', 'Cancelled', 'Lapsed', 'Retained', 'Abandoned', 'Dropped', 'NoShow', 'Fraud', 'Stale', 'Renewed', 'Churned']
categorical_cols = ['gender', 'Gender', 'Contract', 'PaymentMethod', 'InternetService', 'Cat_Feature_Y']

true_variants = ['YES', 'Y', '1', 'TRUE', '1.0']
false_variants = ['NO', 'N', '0', 'FALSE', '0.0']

for col in binary_cols:
    if col in df_step5.columns:
        # Напишите логику маппинга, заполнения модой и приведения к bool
        ...

for col in categorical_cols:
    if col in df_step5.columns:
        # Приведите к нижнему регистру
        df_step5[col] = ...


---

## 🏗 ФИНАЛЬНАЯ СБОРКА: Сквозная функция clean_and_normalise_dataframe 

Теперь объедините разработанный выше код пяти заданий в одну монолитную, гибкую production-функцию. Чтобы код соответствовал Senior-стандартам разработки, мы параметризуем функцию: добавим аргумент `datetime_cols` для динамической передачи названий колонок дат и флаг `drop_dup` для управления удалением дубликатов строк. Именно эту функцию вы перенесете в ядро вашего курсового проекта (`course_project/src/data_pipeline.py`).


In [ ]:
# [MASTER SOLUTION]
def clean_and_normalise_dataframe(df: pd.DataFrame, datetime_cols: list = None, drop_dup: bool = True) -> pd.DataFrame:
    """
    Промышленная ETL-функция для сквозной очистки и нормализации датафрейма профилей.
    Устраняет базовые дефекты качества данных вашего варианта без жесткого хардкода полей.
    """
    df_clean = df.copy()

    # 1. Заголовки (ЗАДАНИЕ 2) 
    df_clean.columns = df_clean.columns.str.strip().str.replace(' ', '_')

    # 2. Текстовые заглушки (ЗАДАНИЕ 3) 
    placeholders = [r'^\s*$', r'^\?$', r'(?i)^null$', r'(?i)^n/a$', r'(?i)^none$']
    for pattern in placeholders:
        df_clean = df_clean.replace(pattern, np.nan, regex=True)

    # 3. Обработка временных полей (Параметрический подход) 
    if datetime_cols is not None:
        for col in datetime_cols:
            if col in df_clean.columns:
                df_clean[col] = pd.to_datetime(df_clean[col], errors='coerce')

    # 4. Числа и заполнение NaN (ЗАДАНИЕ 4) 
    numeric_targets = ['TotalCharges', 'MonthlyCharges', 'Num_Feature_X']
    for col in numeric_targets:
        if col in df_clean.columns:
            df_clean[col] = df_clean[col].astype(str).str.replace(r'[^\d.]', '', regex=True)
            df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
            if 'tenure' in df_clean.columns:
                df_clean[col] = np.where((df_clean['tenure'] == 0) & (df_clean[col].isnull()), 0.0, df_clean[col])
            if df_clean[col].isnull().sum() > 0:
                df_clean[col] = df_clean[col].fillna(df_clean[col].median())

    # 5. Флаги и категории (ЗАДАНИЕ 5) 
    true_variants = ['YES', 'Y', '1', 'TRUE', '1.0']
    false_variants = ['NO', 'N', '0', 'FALSE', '0.0']
    binary_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'Churn', 'TechSupport', 'Target_Flag', 'Exited', 'Default', 'Converted', 'Cancelled', 'Lapsed', 'Retained', 'Abandoned', 'Dropped', 'NoShow', 'Fraud', 'Stale', 'Renewed', 'Churned']
    categorical_cols = ['gender', 'Gender', 'Contract', 'PaymentMethod', 'InternetService', 'Cat_Feature_Y']

    for col in binary_cols:
        if col in df_clean.columns:
            str_col = df_clean[col].astype(str).str.strip().str.upper()
            df_clean.loc[str_col.isin(true_variants), col] = True
            df_clean.loc[str_col.isin(false_variants), col] = False
            if df_clean[col].isnull().sum() > 0:
                df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])
            df_clean[col] = df_clean[col].astype(bool)

    for col in categorical_cols:
        if col in df_clean.columns:
            df_clean[col] = df_clean[col].astype(str).str.strip().str.lower()

    # 6. Удаление полных дубликатов строк (Параметрический подход) 
    if drop_dup:
        df_clean = df_clean.drop_duplicates()

    return df_clean

# Применение собранного пайплайна к исходному сырому датасету
df = clean_and_normalise_dataframe(df_raw, datetime_cols=['Join_Date'])
print(f"Пайплайн успешно запущен! Чистый датафрейм собран. Размерность: {df.shape}")


In [ ]:
# [STUDENT TEMPLATE]
def clean_and_normalise_dataframe(df: pd.DataFrame, datetime_cols: list = None, drop_dup: bool = True) -> pd.DataFrame:
    """
    Промышленная ETL-функция для сквозной очистки и нормализации датафрейма профилей.
    Устраняет все базовые дефекты качества данных вашего варианта.
    """
    # TODO: Соберите архитектуру функции на основе выполненных выше ЗАДАНИЙ 2-5.
    # Учтите использование аргумента datetime_cols для обработки дат и drop_dup для удаления дубликатов строк.
    raise NotImplementedError("Финальная сборка функции не выполнена!")
    
    df_clean = df.copy()
    return df_clean

# Применение собранного пайплайна к исходному сырому датасету
df = clean_and_normalise_dataframe(df_raw, datetime_cols=['Join_Date'])


---

## 🛠 Автоматизированная проверка качества (Autocheck)

Запустите данный валидационный скрипт, чтобы проверить работоспособность и корректность выполнения собранной вами функции `clean_and_normalise_dataframe` по архитектурным критериям.


In [ ]:
def run_autocheck(df_to_check):
    print("🚀 Тестирование собранной функции clean_and_normalise_dataframe...\n" + "-"*45)
    validation_status = True
     
    # Определение имен колонок в зависимости от датасета
    target_col = [c for c in ['Churn', 'Target_Flag', 'Exited', 'Default', 'Converted'] if c in df_to_check.columns]
    charge_col = [c for c in ['TotalCharges', 'Num_Feature_X', 'MonthlyCharges', 'Price', 'Income'] if c in df_to_check.columns]
    date_col = [c for c in ['Join_Date', 'Trans_Date', 'Review_Date'] if c in df_to_check.columns]
    
    if not target_col or not charge_col:
        print("❌ Ошибка: В результирующей таблице не найдены ожидаемые колонки. Проверьте зачистку заголовков.")
        return
        
    t_col = target_col[0]
    c_col = charge_col[0]
    
    # Проверка структуры
    if not isinstance(df_to_check, pd.DataFrame):
        print("❌ Ошибка: Результирующий объект не является валидным DataFrame.")
        validation_status = False
        
    # Валидация чистоты заголовков (Задание 2)
    if any(df_to_check.columns.str.contains(r'^\s|\s$')):
        print("❌ Ошибка: В именах столбцов обнаружены концевые пробелы.")
        validation_status = False
    else:
        print("✅ Заголовки колонок успешно нормализованы.")
        
    # Валидация типов данных и заполнения NaN (Задание 3 и 4)
    if not pd.api.types.is_numeric_dtype(df_to_check[c_col]):
        print(f"❌ Ошибка: Признак {c_col} имеет некорректный нечисловой тип.")
        validation_status = False
    elif df_to_check[c_col].isnull().sum() > 0:
        print(f"❌ Ошибка: В столбце {c_col} обнаружены не устраненные NaN.")
        validation_status = False
    else:
        print(f"✅ Типизация и скрытые пропуски в {c_col} полностью исправлены.")
        
    # Валидация дат
    if date_col and not pd.api.types.is_datetime64_any_dtype(df_to_check[date_col[0]]):
        print(f"❌ Ошибка: Столбец {date_col[0]} не переведен в формат datetime.")
        validation_status = False
    else:
        print(f"✅ Даты успешно приведену к типу datetime.")

    # Валидация дубликатов (Задание 2)
    if df_to_check.duplicated().sum() > 0:
        print("❌ Ошибка: Пайплайн не очищает датасет от строк-дубликатов.")
        validation_status = False
    else:
        print("✅ Дубликаты в таблице полностью отсутствуют.")
        
    # Валидация нормализации логических признаков (Задание 5)
    if df_to_check[t_col].dtype != bool:
        print(f"❌ Ошибка: Целевой признак {t_col} не приведен к булевому типу.")
        validation_status = False
    else:
        print("✅ Кодирование логических флагов стандартизировано.")
    
    print("-" * 45)
    if validation_status:
        print("🎉 ПОЗДРАВЛЯЕМ! Ваш ETL-пайплайн полностью соответствует требованиям программы.")
        print("Вы готовы перенести эту функцию в файл course_project/src/data_pipeline.py! ")
    else:
        print("⚠️ Обнаружены технические дефекты. Проверьте реализацию вашей финальной функции.")

run_autocheck(df)